# AI Reflection and Prompts
**ENGR 010 Group Project — Power Systems Analysis and Monitoring**

---
## Part 1: AI Prompts with Annotations

### Prompt 1 — Project Setup and Code Generation

**Prompt given to Claude:**

> I'm working on a group project for my intro engineering class (ENGR 010) — the Electrical Engineering option. Our teacher gave us a sample data file (`ee_sample_data.py`) that generates a CSV with six months of hourly measurements from three substations (SUB_001, SUB_002, SUB_003), including voltage, current, real power, reactive power, and power factor. The data has daily and seasonal load patterns, plus some injected anomalies.
>
> We need to build a Python application on top of that data. Here are the technical requirements:
> - At least 5 custom functions with docstrings
> - Both for loops AND while loops
> - if/elif/else decision structures
> - try/except error handling
> - Lists and dictionaries as data structures
> - NumPy arrays for signal processing
> - At least 3 different types of Matplotlib plots
> - Code organized into multiple Python files
>
> The analysis features we need:
> - Basic statistics (mean, median, standard deviation) per station
> - Load pattern identification (daily and seasonal)
> - Comparing measurements against grid standards (voltage 0.95–1.05 pu, power factor ≥ 0.90)
> - Power quality indices
> - Basic fault detection
>
> The visualization features:
> - Time series plots of electrical parameters
> - A power triangle visualization
> - An interactive dashboard or creative feature
>
> Please create: (1) `analysis.py` with all the analysis functions, (2) `visualization.py` with all the plotting functions, and (3) a Jupyter notebook `power_system_dashboard.ipynb` that imports from both and walks through all the analysis.

**Annotation:**

*Why did we ask this?*
We had all the requirements from the spec and wanted Claude to help structure the project rather than starting from scratch. We understood the individual concepts from class but weren't sure how to wire them together into multiple files with a real dataset.

*Did Claude answer well?*
Yes. It produced all three files with 7 functions in `analysis.py`, both loop types, `try/except` in `load_data()`, and five different plot types. We checked each requirement against the spec checklist and everything matched. The docstrings made it easy to read through and understand what each function was doing.

*Anything that didn't come out right?*
Claude had no issues with the main project files. The teacher-provided `ee_sample_data.py` had some bugs we caught separately (see Prompt 2), but that was a separate issue from what we asked Claude to build here.

---
### Prompt 2 — Bug Fix in the Teacher-Provided Data Script

**Prompt given to Claude:**

> Can you figure out what is wrong with this code? [shared `ee_sample_data.py`]

**Annotation:**

*Why did we ask this?*
Our teacher gave us `ee_sample_data.py` to generate the CSV. When we ran it we got a pandas deprecation warning, and when we later plotted the March 15th voltage sag we noticed the current values didn't change even though voltage clearly dropped — physically that doesn't make sense since P = V×I. We shared the file with Claude to get a second opinion on what was wrong.

*Did Claude answer well?*
Yes. It caught both bugs and explained each one before touching the code. Bug 1: the frequency alias `'H'` is deprecated in newer pandas — should be `'h'`. Bug 2: the anomaly injection modified `voltage_pu` and `power_factor` but never recalculated the dependent columns (`current_pu`, `reactive_power_mvar`), leaving the data internally inconsistent. The P = V×I explanation actually connected back to what we covered in the EE portion of the class, which helped us verify the fix made sense rather than just accepting it.

---
### Prompt 3 — Interactive Dashboard Using Plotly

**Prompt given to Claude:**

> Looking back at our main notebook, I don't think we actually satisfied the "interactive dashboard" requirement — everything is just static matplotlib plots. Can you create a separate notebook using plotly that has genuinely interactive charts? Things like hover tooltips, zoom, toggling stations on and off in the legend, and a dropdown to switch between parameters would all count. Keep the analysis logic importing from analysis.py and just swap out the visualization layer.

**Annotation:**

*Why did we ask this?*
After reviewing the spec again we realized the "interactive dashboard" requirement wasn't met — our plots were all static images. We decided to use Plotly in a separate file so we wouldn't disturb the main notebook that our teammates had already studied.

*Did Claude answer well?*
Yes, the resulting `dashboard_plotly.ipynb` has six interactive charts including a time series with a built-in parameter dropdown, a four-panel synchronized dashboard, an interactive heatmap, power triangles, a fault detection timeline, and the health score chart. Every chart supports zoom, pan, hover tooltips, and legend toggling out of the box.

*Challenges we ran into:*
Getting Plotly running took more debugging than expected. Our machine had two Python versions (3.12 and 3.14) and pip was installing packages into 3.14 while VS Code's notebook kernel was using 3.12. The fix was running `!{sys.executable} -m pip install plotly nbformat` from inside the notebook itself, which guarantees the install goes to the right interpreter. We also had to restart the kernel after installing before the packages were recognized.

On the code side, the trickiest part was Plotly's subplot legend behavior — by default the legend overlapped with the first subplot title in the multi-panel charts. Plotly's coordinate system for subplots is different from matplotlib's `tight_layout()`, so fixing the spacing required manually tuning the legend `y` position and adding a top margin. It took a few iterations to get right, and we also learned that VS Code holds the notebook in memory and ignores file changes until you close and reopen the tab — which caused confusion early on when our fixes appeared to have no effect.

---
## Part 2: Reflection on Using AI for Programming

**Did you enjoy using AI to help with this project?**

Yes, more than expected. The most useful part wasn't just getting code fast — it was always having a starting point to react to. Starting from a blank file when you're unsure how to structure something is paralyzing. Having a full draft to read through and evaluate felt like a much better use of our time.

**What did AI work well for? What didn't it do as well?**

*Worked well:*
- Generating structure and boilerplate — file organization, docstrings, notebook sections. Tedious stuff that doesn't teach you much.
- On-demand explanations. When we didn't understand something (the rolling z-score for fault detection, `arctan2` for the power triangle, Plotly's trace/layout model), Claude explained it in context of the actual code we were looking at.
- Debugging. Describing a symptom ("current doesn't change when voltage drops") was enough for Claude to diagnose the root cause.

*Didn't work as well:*
- First drafts aren't always correct. The bugs in `ee_sample_data.py` are a good example — if we hadn't caught them ourselves the data would have been physically inconsistent. AI output needs to be checked.
- Environment issues are outside its control. The Python version mismatch between pip and the notebook kernel was something Claude could explain but not fix for us — we had to run the right commands ourselves.
- The `while` loop in `check_grid_standards()` is a bit forced. A `for` loop would be more natural there; the `while` is essentially there to satisfy the project requirement. That's a reminder that AI will meet stated requirements even when the result isn't the cleanest approach.

**Did using AI save you time?**

Yes, significantly. Building three Python files and a notebook with all requirements checked off would have taken many hours across multiple sessions. We had a working draft in one session and spent the rest of our time understanding, testing, and fixing things. The Plotly dashboard would have taken even longer to figure out from scratch given how different it is from Matplotlib.

**Were the concepts used in the AI-generated code things you had already seen and understood?**

Most of them yes — loops, if/else, functions, try/except, Pandas, and basic Matplotlib were all covered in class. A few things were new:

- **Rolling z-scores for fault detection** — `rolling(window=24).mean()` was new syntax, but the concept (flagging readings that are far from recent normal) made sense once Claude explained it. We kept it because it actually works — it correctly flags the March 15th sag.
- **`np.arctan2` for the power triangle** — new function, but the geometry behind it (getting the angle φ in the right quadrant) was familiar from the EE content.
- **Plotly's trace/layout model** — entirely new. Matplotlib thinks in terms of axes you draw on; Plotly thinks in terms of trace objects you add to a figure and a layout you configure separately. It took reading through the code and Claude's explanations to build that mental model.

We were careful not to include anything we couldn't explain if asked about it in the Q&A. If something was unfamiliar we looked it up or asked Claude to explain it before keeping it.